# 🧪 Multi-Agent System Evaluation Checklist

This notebook expands the single-agent checklist into a **multi-agent with tools** evaluation harness for Azure AI Foundry hosted agents.

It validates:
1. Per-agent infrastructure readiness
2. Per-agent identity and RBAC
3. Per-agent tool backend connectivity
4. Router/orchestrator agent selection
5. Per-agent tool selection quality
6. Multi-agent end-to-end execution with live Azure APIs
7. Cross-agent handoff detection
8. Summary scoring and quick-fix command generation

> Update the configuration values in Section 0 before running all cells.

## 0️⃣ Configuration

Define the Foundry project, shared Azure settings, specialist agents, routing tests, multi-agent scenarios, and handoff tests.

In [ ]:
import os
import sys
from pathlib import Path

# Resolve repo root (works whether launched from repo root or notebooks/)
REPO_ROOT = Path(os.getcwd())
if (REPO_ROOT / "notebooks").exists():
    pass  # already at repo root
elif (REPO_ROOT.parent / "notebooks").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
from dotenv import load_dotenv
load_dotenv()

def function_tool(name, description, properties, required=None):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {
                "type": "object",
                "properties": properties,
                "required": required or [],
            },
        },
    }


MULTI_AGENT_CONFIG = {
    "project_endpoint": os.environ.get("AZURE_AI_ENDPOINT", ""),
    "subscription_id": os.environ.get("AZURE_SUBSCRIPTION_ID", ""),
    "acr_name": os.environ.get("AZURE_ACR_NAME", ""),
    "model_deployment": "gpt-4.1-mini",  # ⬇️ CUSTOMIZE: your model deployment name,
    "agents": [
        {
            "name": "monitor-recommendations-agent",
            "description": "Investigates Azure VM inventory, Azure Monitor metrics, and Azure Advisor recommendations to explain operational or cost optimization opportunities.",
            "version": "latest",
            "image_name": "monitor-recommendations-agent",
            "image_tag": "latest",
            "required_roles": [
                {"role": "Reader", "scope": "subscription"},
                {"role": "Monitoring Reader", "scope": "subscription"},
            ],
            "tools": [
                function_tool(
                    "list_vms",
                    "List virtual machines in the subscription or a specific resource group.",
                    {
                        "resource_group": {
                            "type": "string",
                            "description": "Optional resource group name used to scope the VM listing.",
                        }
                    },
                ),
                function_tool(
                    "get_monitor_metrics",
                    "Retrieve Azure Monitor metrics for a VM or another Azure resource.",
                    {
                        "resource_id": {"type": "string", "description": "Full Azure resource ID."},
                        "metric_names": {"type": "string", "description": "Comma-separated metric names."},
                        "time_range_hours": {"type": "integer", "description": "How many hours of history to query."},
                    },
                    required=["resource_id"],
                ),
                function_tool(
                    "get_advisor_recommendations",
                    "List Azure Advisor recommendations for the subscription, optionally filtered by category.",
                    {
                        "category": {
                            "type": "string",
                            "description": "Optional Advisor category such as Cost, HighAvailability, OperationalExcellence, Performance, or Security.",
                        }
                    },
                ),
            ],
            "tool_selection_tests": [
                ("List all virtual machines in my subscription.", "list_vms"),
                ("Show me CPU metrics for VM appvm01 using resource ID /subscriptions/000/resourceGroups/rg-prod/providers/Microsoft.Compute/virtualMachines/appvm01.", "get_monitor_metrics"),
                ("What cost recommendations does Azure Advisor have right now?", "get_advisor_recommendations"),
            ],
            "tool_executors": {
                "list_vms": "Azure Compute API via ComputeManagementClient.virtual_machines.list_all or list(resource_group)",
                "get_monitor_metrics": "Azure Monitor Metrics API via MonitorManagementClient.metrics.list",
                "get_advisor_recommendations": "Azure Advisor API via AdvisorManagementClient.recommendations.list",
            },
        },
        {
            "name": "vm-resize-analyst-agent",
            "description": "Analyzes right-sizing options for Azure virtual machines by checking inventory, regional SKU availability, valid resize targets, and approximate cost differences.",
            "version": "latest",
            "image_name": "vm-resize-analyst-agent",
            "image_tag": "latest",
            "required_roles": [
                {"role": "Reader", "scope": "subscription"},
            ],
            "tools": [
                function_tool(
                    "list_vms",
                    "List virtual machines in the subscription or a specific resource group.",
                    {
                        "resource_group": {
                            "type": "string",
                            "description": "Optional resource group name used to scope the VM listing.",
                        }
                    },
                ),
                function_tool(
                    "get_available_vm_skus",
                    "List available VM sizes in a target Azure region.",
                    {
                        "location": {"type": "string", "description": "Azure region, for example eastus2."},
                        "filter_family": {"type": "string", "description": "Optional substring filter such as Dsv5."},
                    },
                    required=["location"],
                ),
                function_tool(
                    "get_vm_resize_options",
                    "Get the valid resize targets for an existing VM.",
                    {
                        "resource_group": {"type": "string", "description": "Resource group name."},
                        "vm_name": {"type": "string", "description": "Virtual machine name."},
                    },
                    required=["resource_group", "vm_name"],
                ),
                function_tool(
                    "estimate_cost_comparison",
                    "Estimate the monthly cost delta between the current and target VM SKUs using the Azure Retail Prices API.",
                    {
                        "current_sku": {"type": "string", "description": "Current VM SKU, for example Standard_D2s_v3."},
                        "target_sku": {"type": "string", "description": "Target VM SKU."},
                        "region": {"type": "string", "description": "Azure region for price lookup."},
                    },
                    required=["current_sku", "target_sku", "region"],
                ),
            ],
            "tool_selection_tests": [
                ("Show me every VM in resource group rg-prod.", "list_vms"),
                ("What VM sizes are available in eastus2 for the Dsv5 family?", "get_available_vm_skus"),
                ("Get resize options for VM appvm01 in resource group rg-prod.", "get_vm_resize_options"),
                ("Compare monthly cost for Standard_D2s_v3 vs Standard_D4s_v3 in eastus2.", "estimate_cost_comparison"),
            ],
            "tool_executors": {
                "list_vms": "Azure Compute API via ComputeManagementClient.virtual_machines.list_all or list(resource_group)",
                "get_available_vm_skus": "Azure Compute API via ComputeManagementClient.virtual_machine_sizes.list",
                "get_vm_resize_options": "Azure Compute API via ComputeManagementClient.virtual_machines.list_available_sizes",
                "estimate_cost_comparison": "Azure Retail Prices API via https://prices.azure.com/api/retail/prices",
            },
        },
    ],
    "routing_tests": [
        ("Show me Advisor recommendations for my Azure VMs.", "monitor-recommendations-agent"),
        ("Which VM sizes are available in eastus2 for a resize exercise?", "vm-resize-analyst-agent"),
        ("Get CPU metrics for VM appvm01 so I can see if it is overloaded.", "monitor-recommendations-agent"),
        ("Estimate whether moving from Standard_D2s_v3 to Standard_D4s_v3 changes cost a lot.", "vm-resize-analyst-agent"),
    ],
    "e2e_scenarios": [
        {
            "name": "Advisor recommendations scenario",
            "prompt": "Show me Azure Advisor cost recommendations for my subscription and summarize the top findings.",
            "expected_agent": "monitor-recommendations-agent",
            "expected_tool": "get_advisor_recommendations",
            "validation_keywords": ["advisor", "recommendation"],
        },
        {
            "name": "Regional SKU discovery scenario",
            "prompt": "What VM sizes are available in eastus2 for the Dsv5 family?",
            "expected_agent": "vm-resize-analyst-agent",
            "expected_tool": "get_available_vm_skus",
            "validation_keywords": ["eastus2", "vm", "size"],
        },
        {
            "name": "Cost comparison scenario",
            "prompt": "Compare the monthly cost of Standard_D2s_v3 and Standard_D4s_v3 in eastus2.",
            "expected_agent": "vm-resize-analyst-agent",
            "expected_tool": "estimate_cost_comparison",
            "validation_keywords": ["monthly", "cost", "eastus2"],
        },
    ],
    "handoff_tests": [
        {
            "name": "Metrics then resize recommendation",
            "prompt": "Identify VMs with sustained CPU pressure and then recommend resize targets and cost deltas for the most impacted VM.",
            "expected_primary_agent": "monitor-recommendations-agent",
            "expected_secondary_agents": ["vm-resize-analyst-agent"],
        },
        {
            "name": "Advisor savings follow-up",
            "prompt": "Find cost optimization recommendations from Advisor and then estimate the monthly savings for the suggested right-size VM option.",
            "expected_primary_agent": "monitor-recommendations-agent",
            "expected_secondary_agents": ["vm-resize-analyst-agent"],
        },
    ],
}

print(f"Configured {len(MULTI_AGENT_CONFIG['agents'])} agents for evaluation.")
for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    tool_names = [tool["function"]["name"] for tool in agent_cfg["tools"]]
    print(f"- {agent_cfg['name']}: {tool_names}")

## 1️⃣ Imports & Helpers

Load the Azure SDKs, OpenAI client, result trackers, helper functions, router helpers, and live tool executors.

In [ ]:
import json
import re
import subprocess
from collections import defaultdict
from datetime import datetime, timedelta, timezone

import requests
from tabulate import tabulate

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.mgmt.advisor import AdvisorManagementClient
from azure.mgmt.compute import ComputeManagementClient
from azure.mgmt.monitor import MonitorManagementClient
from openai import AzureOpenAI


credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)
project_client = AIProjectClient(endpoint=MULTI_AGENT_CONFIG["project_endpoint"], credential=credential)
openai_client = project_client.get_openai_client()

results = []
agent_results = {agent["name"]: [] for agent in MULTI_AGENT_CONFIG["agents"]}
system_metrics = {
    "routing": {"passed": 0, "total": 0},
    "handoff": {"passed": 0, "total": 0},
}


def check(name, passed, detail=""):
    icon = "✅" if passed else "❌"
    results.append({"test": name, "passed": passed, "detail": detail, "section": name.split(":", 1)[0] if ":" in name else "system"})
    print(f"{icon} {name}" + (f" — {detail}" if detail else ""))
    return passed


def agent_check(agent_name, test_name, passed, detail=""):
    icon = "✅" if passed else "❌"
    record = {
        "agent": agent_name,
        "test": test_name,
        "passed": passed,
        "detail": detail,
        "section": test_name.split(":", 1)[0] if ":" in test_name else "other",
    }
    agent_results.setdefault(agent_name, []).append(record)
    print(f"{icon} [{agent_name}] {test_name}" + (f" — {detail}" if detail else ""))
    return passed


def az_cli(cmd):
    result = subprocess.run(
        f"az {cmd} -o json",
        shell=True,
        capture_output=True,
        text=True,
        timeout=180,
    )
    if result.returncode != 0:
        return None
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return result.stdout.strip()


def model_to_dict(model):
    if model is None:
        return {}
    if isinstance(model, dict):
        return model
    if hasattr(model, "as_dict"):
        try:
            return model.as_dict()
        except Exception:
            pass
    data = getattr(model, "_data", None)
    if isinstance(data, dict):
        return data
    return {}


def resolve_scope(scope_value, subscription_id):
    if not scope_value:
        return f"/subscriptions/{subscription_id}"
    if scope_value == "subscription":
        return f"/subscriptions/{subscription_id}"
    return scope_value.format(subscription_id=subscription_id)


def get_agent_config(agent_name):
    for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
        if agent_cfg["name"] == agent_name:
            return agent_cfg
    raise KeyError(f"Unknown agent: {agent_name}")


class AgentRouter:
    def __init__(self, agents, openai_client, model_name):
        self.agents = agents
        self.openai_client = openai_client
        self.model_name = model_name
        self.system_prompt = (
            "You are a specialist agent router. Always choose exactly one specialist agent as a tool call. "
            "Select the agent whose description most directly matches the user request."
        )

    def build_tools(self):
        return [
            {
                "type": "function",
                "function": {
                    "name": agent_cfg["name"],
                    "description": agent_cfg["description"],
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "reason": {
                                "type": "string",
                                "description": "Why this specialist agent should handle the request.",
                            }
                        },
                        "required": [],
                    },
                },
            }
            for agent_cfg in self.agents
        ]

    def route(self, prompt):
        _, _, tool_calls = run_tool_choice(self.system_prompt, prompt, self.build_tools(), model=self.model_name)
        return tool_call_name(tool_calls), tool_calls


def run_tool_choice(system_prompt, user_prompt, tools, model=None):
    response = openai_client.chat.completions.create(
        model=model or MULTI_AGENT_CONFIG["model_deployment"],
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        tools=tools,
        tool_choice="auto",
    )
    message = response.choices[0].message
    tool_calls = getattr(message, "tool_calls", None) or []
    return response, message, tool_calls


def tool_call_name(tool_calls):
    return tool_calls[0].function.name if tool_calls else None


def tool_call_args(tool_call):
    raw = getattr(tool_call.function, "arguments", "{}") or "{}"
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}


def assistant_message_from_tool_calls(message, tool_calls):
    return {
        "role": "assistant",
        "content": message.content or "",
        "tool_calls": [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments,
                },
            }
            for tool_call in tool_calls
        ],
    }


def get_compute_client():
    return ComputeManagementClient(credential, MULTI_AGENT_CONFIG["subscription_id"])


def get_monitor_client():
    return MonitorManagementClient(credential, MULTI_AGENT_CONFIG["subscription_id"])


def get_advisor_client():
    return AdvisorManagementClient(credential, MULTI_AGENT_CONFIG["subscription_id"])


def fetch_retail_price(arm_sku_name, region):
    response = requests.get(
        "https://prices.azure.com/api/retail/prices",
        params={
            "$filter": f"serviceName eq 'Virtual Machines' and armSkuName eq '{arm_sku_name}' and armRegionName eq '{region}' and priceType eq 'Consumption'"
        },
        timeout=30,
    )
    response.raise_for_status()
    data = response.json()
    items = data.get("Items", [])
    meter = next(
        (
            item
            for item in items
            if item.get("unitOfMeasure") in {"1 Hour", "Hour"} and "Spot" not in item.get("skuName", "")
        ),
        items[0] if items else None,
    )
    if not meter:
        raise ValueError(f"No retail price found for {arm_sku_name} in {region}")
    hourly = float(meter.get("retailPrice", 0.0))
    return {
        "sku": arm_sku_name,
        "region": region,
        "currency_code": meter.get("currencyCode"),
        "hourly_price": hourly,
        "monthly_estimate": round(hourly * 730, 2),
        "product_name": meter.get("productName"),
    }


def execute_tool(tool_name, tool_args):
    try:
        if tool_name == "list_vms":
            compute_client = get_compute_client()
            resource_group = tool_args.get("resource_group")
            vm_iterable = (
                compute_client.virtual_machines.list(resource_group)
                if resource_group
                else compute_client.virtual_machines.list_all()
            )
            vms = []
            for vm in list(vm_iterable)[:25]:
                vm_id = getattr(vm, "id", "")
                rg_name = resource_group or (vm_id.split("/")[4] if vm_id and len(vm_id.split("/")) > 4 else None)
                vms.append(
                    {
                        "name": getattr(vm, "name", None),
                        "resource_group": rg_name,
                        "location": getattr(vm, "location", None),
                        "vm_id": vm_id,
                    }
                )
            return {"count": len(vms), "items": vms}

        if tool_name == "get_advisor_recommendations":
            advisor_client = get_advisor_client()
            category = tool_args.get("category")
            recommendations = []
            for rec in list(advisor_client.recommendations.list())[:25]:
                rec_data = model_to_dict(rec)
                short_desc = rec_data.get("short_description") or {}
                rec_category = rec_data.get("category") or rec_data.get("properties", {}).get("category")
                if category and str(rec_category).lower() != str(category).lower():
                    continue
                recommendations.append(
                    {
                        "category": rec_category,
                        "impact": rec_data.get("impact") or rec_data.get("properties", {}).get("impact"),
                        "risk": rec_data.get("risk") or rec_data.get("properties", {}).get("risk"),
                        "problem": short_desc.get("problem") or rec_data.get("name"),
                        "solution": short_desc.get("solution"),
                        "resource_id": rec_data.get("resource_metadata", {}).get("resource_id") or rec_data.get("id"),
                    }
                )
            return {"count": len(recommendations), "items": recommendations[:10]}

        if tool_name == "get_monitor_metrics":
            monitor_client = get_monitor_client()
            resource_id = tool_args["resource_id"]
            metric_names = tool_args.get("metric_names", "Percentage CPU")
            time_range_hours = int(tool_args.get("time_range_hours", 6))
            end_time = datetime.now(timezone.utc)
            start_time = end_time - timedelta(hours=time_range_hours)
            timespan = f"{start_time.isoformat()}/{end_time.isoformat()}"
            metric_response = monitor_client.metrics.list(
                resource_uri=resource_id,
                timespan=timespan,
                interval="PT1H",
                metricnames=metric_names,
                aggregation="Average,Minimum,Maximum",
            )
            series_payload = []
            for metric in getattr(metric_response, "value", [])[:10]:
                metric_name = getattr(getattr(metric, "name", None), "value", None)
                for series in getattr(metric, "timeseries", [])[:3]:
                    datapoints = []
                    for point in getattr(series, "data", [])[-5:]:
                        datapoints.append(
                            {
                                "timestamp": getattr(point, "time_stamp", None).isoformat() if getattr(point, "time_stamp", None) else None,
                                "average": getattr(point, "average", None),
                                "minimum": getattr(point, "minimum", None),
                                "maximum": getattr(point, "maximum", None),
                            }
                        )
                    series_payload.append({"metric": metric_name, "data": datapoints})
            return {"resource_id": resource_id, "metric_names": metric_names, "series": series_payload}

        if tool_name == "get_available_vm_skus":
            compute_client = get_compute_client()
            location = tool_args["location"]
            filter_family = (tool_args.get("filter_family") or "").lower()
            sizes = []
            for size in list(compute_client.virtual_machine_sizes.list(location)):
                name = getattr(size, "name", "")
                if filter_family and filter_family not in name.lower():
                    continue
                sizes.append(
                    {
                        "name": name,
                        "number_of_cores": getattr(size, "number_of_cores", None),
                        "memory_in_mb": getattr(size, "memory_in_mb", None),
                        "max_data_disk_count": getattr(size, "max_data_disk_count", None),
                    }
                )
            return {"location": location, "count": len(sizes), "items": sizes[:20]}

        if tool_name == "get_vm_resize_options":
            compute_client = get_compute_client()
            resource_group = tool_args["resource_group"]
            vm_name = tool_args["vm_name"]
            sizes = list(compute_client.virtual_machines.list_available_sizes(resource_group, vm_name))
            return {
                "vm_name": vm_name,
                "resource_group": resource_group,
                "count": len(sizes),
                "items": [
                    {
                        "name": getattr(size, "name", None),
                        "number_of_cores": getattr(size, "number_of_cores", None),
                        "memory_in_mb": getattr(size, "memory_in_mb", None),
                    }
                    for size in sizes[:20]
                ],
            }

        if tool_name == "estimate_cost_comparison":
            current_sku = tool_args["current_sku"]
            target_sku = tool_args["target_sku"]
            region = tool_args["region"]
            current_price = fetch_retail_price(current_sku, region)
            target_price = fetch_retail_price(target_sku, region)
            monthly_delta = round(target_price["monthly_estimate"] - current_price["monthly_estimate"], 2)
            return {
                "region": region,
                "current": current_price,
                "target": target_price,
                "monthly_delta": monthly_delta,
                "cheaper_option": current_sku if monthly_delta > 0 else target_sku,
            }

        raise ValueError(f"Unsupported tool executor: {tool_name}")
    except Exception as exc:
        return {"error": str(exc), "tool": tool_name, "args": tool_args}


router = AgentRouter(MULTI_AGENT_CONFIG["agents"], openai_client, MULTI_AGENT_CONFIG["model_deployment"])


def run_router(prompt):
    return router.route(prompt)


def final_response_with_tool_result(agent_cfg, user_prompt, message, tool_calls, tool_outputs):
    follow_up_messages = [
        {
            "role": "system",
            "content": f"You are {agent_cfg['name']}. {agent_cfg['description']} Use the tool outputs to answer clearly and concisely.",
        },
        {"role": "user", "content": user_prompt},
        assistant_message_from_tool_calls(message, tool_calls),
    ]
    for output in tool_outputs:
        follow_up_messages.append(
            {
                "role": "tool",
                "tool_call_id": output["tool_call_id"],
                "name": output["name"],
                "content": json.dumps(output["payload"], default=str),
            }
        )
    final_response = openai_client.chat.completions.create(
        model=MULTI_AGENT_CONFIG["model_deployment"],
        messages=follow_up_messages,
    )
    return final_response.choices[0].message.content or ""


def keyword_hits(text, keywords):
    lowered = (text or "").lower()
    return [keyword for keyword in keywords if keyword.lower() in lowered]


print(f"Helpers loaded for {len(MULTI_AGENT_CONFIG['agents'])} agents.")
print(f"Using model deployment: {MULTI_AGENT_CONFIG['model_deployment']}")

## 2️⃣ Per-Agent Infrastructure Checks

Validate that each hosted agent image exists in ACR, the agent is registered in Foundry, and the selected version is active.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 2: PER-AGENT INFRASTRUCTURE")
print("=" * 70 + "\n")

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    image_name = agent_cfg["image_name"]
    image_tag = agent_cfg["image_tag"]
    acr_name = MULTI_AGENT_CONFIG["acr_name"]


    if acr_name and not acr_name.startswith('<'):
        tags = az_cli(f"acr repository show-tags --name {acr_name} --repository {image_name}")
        acr_found = bool(tags and image_tag in tags)
        agent_check(
            agent_name,
            "infra: ACR image exists",
            acr_found,
            f"{image_name}:{image_tag} in {acr_name}" if acr_found else f"Missing image/tag in ACR {acr_name}",
        )
    else:
        agent_check(agent_name, "infra: ACR image exists", None, "Skipped -- AZURE_ACR_NAME not configured")

    try:
        agent_obj = project_client.agents.get(agent_name=agent_name)
        agent_data = model_to_dict(agent_obj)
        agent_cfg["_agent_id"] = agent_data.get("id") or getattr(agent_obj, "id", None)
        agent_check(agent_name, "infra: Agent registered in Foundry", True, f"id={agent_cfg.get('_agent_id')}")
    except Exception as exc:
        agent_check(agent_name, "infra: Agent registered in Foundry", False, str(exc))
        agent_cfg["_agent_id"] = None

    try:
        target_version = agent_cfg["version"]
        if target_version == "latest":
            versions = list(project_client.agents.list_versions(agent_name=agent_name))
            if versions:
                latest_version_data = model_to_dict(versions[0])
                target_version = latest_version_data.get("version") or getattr(versions[0], "version", "latest")
        version_obj = project_client.agents.get_version(agent_name=agent_name, agent_version=target_version)
        version_data = model_to_dict(version_obj)
        status = version_data.get("status") or version_data.get("state") or "unknown"
        instance_identity = version_data.get("instance_identity") or {}
        agent_cfg["_resolved_version"] = target_version
        agent_cfg["_principal_id"] = instance_identity.get("principal_id") or instance_identity.get("principalId") or ""
        agent_check(agent_name, "infra: Agent version active", str(status).lower() == "active", f"version={target_version}; status={status}")
        agent_check(agent_name, "infra: Principal ID captured", bool(agent_cfg["_principal_id"]), agent_cfg["_principal_id"] or "No principal_id exposed")
    except Exception as exc:
        agent_cfg["_resolved_version"] = None
        agent_cfg["_principal_id"] = ""
        agent_check(agent_name, "infra: Agent version active", False, str(exc))

## 3️⃣ Per-Agent Identity & RBAC

Verify each hosted agent's managed identity and confirm its required role assignments are present at the expected scopes.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 3: PER-AGENT IDENTITY & RBAC")
print("=" * 70 + "\n")

subscription_id = MULTI_AGENT_CONFIG["subscription_id"]

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    principal_id = agent_cfg.get("_principal_id", "")
    agent_cfg["_missing_roles"] = []

    if not principal_id:
        agent_check(agent_name, "rbac: Principal ID available", False, "Could not retrieve principal_id from the active version")
        continue

    agent_check(agent_name, "rbac: Principal ID available", True, principal_id)
    assignments = az_cli(
        f'role assignment list --assignee "{principal_id}" --subscription {subscription_id} --query "[].{{role:roleDefinitionName,scope:scope}}"'
    )
    assignments = assignments or []
    print(f"Assigned roles for {agent_name}: {assignments}\n")

    for requirement in agent_cfg["required_roles"]:
        required_role = requirement["role"]
        required_scope = resolve_scope(requirement["scope"], subscription_id)
        matching = [
            assignment
            for assignment in assignments
            if assignment.get("role") == required_role and str(assignment.get("scope", "")).lower().startswith(required_scope.lower())
        ]
        passed = bool(matching)
        detail = (
            f"role={required_role}; scope={required_scope}"
            if passed
            else f'MISSING — az role assignment create --assignee "{principal_id}" --role "{required_role}" --scope "{required_scope}"'
        )
        agent_check(agent_name, f"rbac: {required_role}", passed, detail)
        if not passed:
            agent_cfg["_missing_roles"].append({"role": required_role, "scope": required_scope})

## 4️⃣ Per-Agent Tool Connectivity

Probe only the live backend APIs needed by each agent's tool set.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 4: PER-AGENT TOOL CONNECTIVITY")
print("=" * 70 + "\n")

probe_region = "eastus2"

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    tool_names = {tool["function"]["name"] for tool in agent_cfg["tools"]}

    if tool_names.intersection({"list_vms", "get_available_vm_skus", "get_vm_resize_options"}):
        try:
            compute_client = get_compute_client()
            sizes = list(compute_client.virtual_machine_sizes.list(probe_region))
            agent_check(
                agent_name,
                "tool_conn: Compute API accessible",
                len(sizes) > 0,
                f"{len(sizes)} VM sizes returned from {probe_region}",
            )
        except Exception as exc:
            agent_check(agent_name, "tool_conn: Compute API accessible", False, str(exc)[:180])

    if "get_monitor_metrics" in tool_names:
        try:
            monitor_client = get_monitor_client()
            end_time = datetime.now(timezone.utc)
            start_time = end_time - timedelta(hours=1)
            activity_filter = f"eventTimestamp ge '{start_time.isoformat()}' and eventTimestamp le '{end_time.isoformat()}'"
            events = list(monitor_client.activity_logs.list(filter=activity_filter))[:5]
            agent_check(
                agent_name,
                "tool_conn: Monitor API accessible",
                True,
                f"Connected successfully ({len(events)} recent activity log events)",
            )
        except Exception as exc:
            agent_check(agent_name, "tool_conn: Monitor API accessible", False, str(exc)[:180])

    if "get_advisor_recommendations" in tool_names:
        try:
            advisor_client = get_advisor_client()
            recommendations = list(advisor_client.recommendations.list())[:5]
            agent_check(
                agent_name,
                "tool_conn: Advisor API accessible",
                True,
                f"Connected successfully ({len(recommendations)} recommendations sampled)",
            )
        except Exception as exc:
            agent_check(agent_name, "tool_conn: Advisor API accessible", False, str(exc)[:180])

    if "estimate_cost_comparison" in tool_names:
        try:
            price_data = fetch_retail_price("Standard_D2s_v3", probe_region)
            agent_check(
                agent_name,
                "tool_conn: Retail Prices API accessible",
                price_data["hourly_price"] > 0,
                f"{price_data['sku']} @ {price_data['hourly_price']} {price_data['currency_code']} per hour",
            )
        except Exception as exc:
            agent_check(agent_name, "tool_conn: Retail Prices API accessible", False, str(exc)[:180])

## 5️⃣ Agent Router / Orchestrator Test

Represent each specialist agent as a tool and verify the routing model selects the right specialist for each prompt.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 5: ROUTER / ORCHESTRATOR")
print("=" * 70 + "\n")

routing_total = len(MULTI_AGENT_CONFIG["routing_tests"])
routing_passed = 0

for prompt, expected_agent in MULTI_AGENT_CONFIG["routing_tests"]:
    selected_agent, tool_calls = run_router(prompt)
    passed = selected_agent == expected_agent
    routing_passed += int(passed)
    system_metrics["routing"]["total"] += 1
    system_metrics["routing"]["passed"] += int(passed)
    detail = f"expected={expected_agent}; actual={selected_agent}; raw_calls={len(tool_calls)}"
    check(f"routing: {prompt[:70]}", passed, detail)

routing_accuracy = (routing_passed / routing_total) if routing_total else 0.0
print(f"\nRouting accuracy: {routing_passed}/{routing_total} = {routing_accuracy:.1%}")

## 6️⃣ Per-Agent Tool Selection

For each specialist agent, verify that the model selects the correct function-calling tool for representative prompts.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 6: PER-AGENT TOOL SELECTION")
print("=" * 70 + "\n")

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    tests = agent_cfg["tool_selection_tests"]
    passed_count = 0
    system_prompt = (
        f"You are {agent_name}. {agent_cfg['description']} "
        "Always choose the single best tool for the user's request and prefer tools over guessing."
    )
    for prompt, expected_tool in tests:
        _, _, tool_calls = run_tool_choice(system_prompt, prompt, agent_cfg["tools"])
        selected_tool = tool_call_name(tool_calls)
        passed = selected_tool == expected_tool
        passed_count += int(passed)
        agent_check(
            agent_name,
            f"tool_select: {prompt[:60]}",
            passed,
            f"expected={expected_tool}; actual={selected_tool}",
        )
    print(f"{agent_name} tool selection accuracy: {passed_count}/{len(tests)} = {(passed_count / len(tests)) if tests else 0:.1%}\n")

## 7️⃣ Multi-Agent E2E Scenarios

Run the full loop: router selection → specialist tool selection → live tool execution → final grounded response validation.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 7: MULTI-AGENT END-TO-END")
print("=" * 70 + "\n")

for scenario in MULTI_AGENT_CONFIG["e2e_scenarios"]:
    scenario_name = scenario["name"]
    prompt = scenario["prompt"]
    expected_agent = scenario["expected_agent"]
    expected_tool = scenario["expected_tool"]
    validation_keywords = scenario["validation_keywords"]

    selected_agent, _ = run_router(prompt)
    if selected_agent != expected_agent:
        detail = f"Router mismatch: expected={expected_agent}; actual={selected_agent}"
        check(f"e2e: {scenario_name}", False, detail)
        agent_check(expected_agent, f"e2e: {scenario_name}", False, detail)
        continue

    agent_cfg = get_agent_config(selected_agent)
    system_prompt = (
        f"You are {selected_agent}. {agent_cfg['description']} "
        "Use tools whenever they are needed to answer with current Azure data."
    )
    _, message, tool_calls = run_tool_choice(system_prompt, prompt, agent_cfg["tools"])
    selected_tool = tool_call_name(tool_calls)
    if selected_tool != expected_tool:
        detail = f"Tool mismatch: expected={expected_tool}; actual={selected_tool}"
        check(f"e2e: {scenario_name}", False, detail)
        agent_check(expected_agent, f"e2e: {scenario_name}", False, detail)
        continue

    tool_outputs = []
    for tool_call in tool_calls:
        payload = execute_tool(tool_call.function.name, tool_call_args(tool_call))
        tool_outputs.append(
            {
                "tool_call_id": tool_call.id,
                "name": tool_call.function.name,
                "payload": payload,
            }
        )

    final_text = final_response_with_tool_result(agent_cfg, prompt, message, tool_calls, tool_outputs)
    matched_keywords = keyword_hits(final_text, validation_keywords)
    passed = len(matched_keywords) == len(validation_keywords)
    detail = (
        f"agent={selected_agent}; tool={selected_tool}; matched_keywords={matched_keywords}; "
        f"response_preview={final_text[:180]}"
    )
    check(f"e2e: {scenario_name}", passed, detail)
    agent_check(expected_agent, f"e2e: {scenario_name}", passed, detail)

## 8️⃣ Cross-Agent Handoff Tests

Evaluate whether the system recognizes prompts that need more than one specialist agent and identifies the expected primary and secondary agents.

In [ ]:
print("\n" + "=" * 70)
print("SECTION 8: CROSS-AGENT HANDOFF")
print("=" * 70 + "\n")

available_agents = [
    {"name": agent_cfg["name"], "description": agent_cfg["description"]}
    for agent_cfg in MULTI_AGENT_CONFIG["agents"]
]
handoff_system_prompt = (
    "You are a multi-agent orchestration planner. "
    "Given a user prompt and the available specialist agents, decide whether the work should be handled by one or multiple agents. "
    "Return JSON only with keys: needs_handoff (boolean), primary_agent (string), secondary_agents (array of strings), rationale (string)."
)

for test in MULTI_AGENT_CONFIG["handoff_tests"]:
    response = openai_client.chat.completions.create(
        model=MULTI_AGENT_CONFIG["model_deployment"],
        response_format={"type": "json_object"},
        messages=[
            {
                "role": "system",
                "content": handoff_system_prompt + " Available agents: " + json.dumps(available_agents),
            },
            {"role": "user", "content": test["prompt"]},
        ],
    )
    content = response.choices[0].message.content or "{}"
    try:
        plan = json.loads(content)
    except json.JSONDecodeError:
        plan = {"needs_handoff": False, "primary_agent": None, "secondary_agents": [], "rationale": content}

    expected_primary = test["expected_primary_agent"]
    expected_secondary = set(test.get("expected_secondary_agents", []))
    actual_primary = plan.get("primary_agent")
    actual_secondary = set(plan.get("secondary_agents", []))
    needs_handoff = bool(plan.get("needs_handoff"))
    passed = needs_handoff and actual_primary == expected_primary and expected_secondary.issubset(actual_secondary)

    system_metrics["handoff"]["total"] += 1
    system_metrics["handoff"]["passed"] += int(passed)
    detail = (
        f"expected_primary={expected_primary}; actual_primary={actual_primary}; "
        f"expected_secondary={sorted(expected_secondary)}; actual_secondary={sorted(actual_secondary)}"
    )
    check(f"handoff: {test['name']}", passed, detail)

## 9️⃣ Summary Report

Summarize total system results, per-agent section scores, and routing accuracy, then print failure details.

In [ ]:
print("\n" + "═" * 80)
print("SUMMARY REPORT")
print("═" * 80)
print(f"Project endpoint: {MULTI_AGENT_CONFIG['project_endpoint']}")
print(f"Model deployment: {MULTI_AGENT_CONFIG['model_deployment']}")
print(f"Timestamp: {datetime.now().isoformat()}")
print("─" * 80)

all_agent_records = [record for records in agent_results.values() for record in records]
all_records = results + all_agent_records
total_passed = sum(1 for record in all_records if record["passed"])
total_failed = sum(1 for record in all_records if not record["passed"])
total_count = len(all_records)
overall_score = (total_passed / total_count) if total_count else 0.0

print(f"System-wide score: {total_passed}/{total_count} = {overall_score:.1%}")
print(f"Routing accuracy: {system_metrics['routing']['passed']}/{system_metrics['routing']['total']} = {(system_metrics['routing']['passed'] / system_metrics['routing']['total']) if system_metrics['routing']['total'] else 0:.1%}")
print(f"Handoff accuracy: {system_metrics['handoff']['passed']}/{system_metrics['handoff']['total']} = {(system_metrics['handoff']['passed'] / system_metrics['handoff']['total']) if system_metrics['handoff']['total'] else 0:.1%}")
print()

def section_score(agent_name, section_prefix):
    section_records = [record for record in agent_results.get(agent_name, []) if record["section"] == section_prefix]
    passed = sum(1 for record in section_records if record["passed"])
    total = len(section_records)
    return f"{passed}/{total}" if total else "0/0"

summary_rows = []
for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    summary_rows.append(
        {
            "agent_name": agent_name,
            "infra_score": section_score(agent_name, "infra"),
            "rbac_score": section_score(agent_name, "rbac"),
            "tool_conn_score": section_score(agent_name, "tool_conn"),
            "tool_select_score": section_score(agent_name, "tool_select"),
            "e2e_score": section_score(agent_name, "e2e"),
        }
    )

print(tabulate(summary_rows, headers="keys", tablefmt="github"))

failing_rows = []
for record in all_records:
    if not record["passed"]:
        failing_rows.append(
            {
                "scope": record.get("agent", "system"),
                "test": record["test"],
                "detail": record.get("detail", ""),
            }
        )

if failing_rows:
    print("\nFailures")
    print(tabulate(failing_rows, headers="keys", tablefmt="github"))
else:
    print("\nNo failures detected.")

## 🔟 Quick Fix Commands

Generate RBAC remediation commands for missing assignments and generic image rebuild / redeploy commands for each hosted agent.

In [ ]:
print("# QUICK FIX COMMANDS\n")

subscription_id = MULTI_AGENT_CONFIG["subscription_id"]
acr_name = MULTI_AGENT_CONFIG["acr_name"]
project_endpoint = MULTI_AGENT_CONFIG["project_endpoint"]

for agent_cfg in MULTI_AGENT_CONFIG["agents"]:
    agent_name = agent_cfg["name"]
    principal_id = agent_cfg.get("_principal_id") or "<principal-id>"
    print(f"## {agent_name}")

    missing_roles = agent_cfg.get("_missing_roles", [])
    if missing_roles:
        print("# RBAC fixes")
        for role in missing_roles:
            print(
                f'az role assignment create --assignee "{principal_id}" --role "{role["role"]}" --scope "{role["scope"]}"'
            )
    else:
        print("# RBAC fixes\n# No missing roles detected.")

    image_ref = f"{agent_cfg['image_name']}:{agent_cfg['image_tag']}"
    print("\n# Rebuild image")
    print(f"az acr build --registry {acr_name} --image {image_ref} .")
    print("\n# Redeploy / publish a new hosted-agent version")
    print(f"# Use your existing deployment automation with project endpoint: {project_endpoint}")
    print(f"# Agent: {agent_name}; image: {acr_name}.azurecr.io/{image_ref}; requested version: {agent_cfg['version']}")
    print("\n")